In [ ]:
# ============================================================
# FRAUD DETECTION SYSTEM - PHASE 1: SETUP & EDA
# Dataset: IEEE-CIS Fraud Detection (Kaggle)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# Plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print("✅ All libraries imported successfully!")

In [ ]:
# ─────────────────────────────────────────
# Make sure train_transaction.csv and train_identity.csv
# are inside your data/ folder
# Download from: https://www.kaggle.com/c/ieee-fraud-detection/data
# ─────────────────────────────────────────

train_transaction = pd.read_csv('../data/train_transaction.csv')
train_identity    = pd.read_csv('../data/train_identity.csv')

# Merge on TransactionID (left join keeps all transactions)
df = pd.merge(train_transaction, train_identity,
              on='TransactionID', how='left')

# Free memory
del train_transaction, train_identity

print(f"✅ Dataset loaded successfully!")
print(f"📊 Shape: {df.shape}")
print(f"📋 Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

In [ ]:
print("=" * 60)
print("FIRST 5 ROWS")
print("=" * 60)
df.head()

In [ ]:
print("=" * 60)
print("DATASET INFO")
print("=" * 60)
df.info(verbose=False)

print("\n")
print("=" * 60)
print("BASIC STATISTICS")
print("=" * 60)
df.describe()
    

In [ ]:
print("=" * 60)
print("TARGET VARIABLE: isFraud")
print("=" * 60)

fraud_counts = df['isFraud'].value_counts()
fraud_pct    = df['isFraud'].value_counts(normalize=True) * 100

print(f"\n🔴     Fraudulent Transactions   : {fraud_counts[1]:,} ({fraud_pct[1]:.2f}%)")
print(f"🟢 Legitimate Transactions   : {fraud_counts[0]:,} ({fraud_pct[0]:.2f}%)")
print(f"\n⚠️  Class Imbalance Ratio     : 1 : {int(fraud_counts[0]/fraud_counts[1])}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar Chart
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Legitimate (0)', 'Fraud (1)'],
            fraud_counts.values,
            color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Transaction Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(fraud_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie Chart
axes[1].pie(fraud_counts.values,
            labels=['Legitimate', 'Fraud'],
            colors=colors,
            autopct='%1.2f%%',
            startangle=90,
            explode=(0, 0.1))
axes[1].set_title('Fraud vs Legitimate (%)', fontsize=14, fontweight='bold')

plt.suptitle('⚠️ Highly Imbalanced Dataset — Will Need SMOTE in Phase 3',
             fontsize=13, color='red', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("MISSING VALUE ANALYSIS")
print("=" * 60)

# Calculate missing %
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %':     (df.isnull().sum() / len(df)) * 100
}).sort_values('Missing %', ascending=False)

missing = missing[missing['Missing Count'] > 0]

print(f"\nTotal columns with missing values : {len(missing)}")
print(f"Columns with >50% missing          : {len(missing[missing['Missing %'] > 50])}")
print(f"Columns with >90% missing          : {len(missing[missing['Missing %'] > 90])}")

# Top 20
print("\nTop 20 columns with most missing values:")
print(missing.head(20).to_string())

# Visualize top 30
plt.figure(figsize=(14, 6))
top_missing = missing.head(30)
plt.barh(top_missing.index[::-1],
         top_missing['Missing %'][::-1],
         color='#e74c3c')
plt.axvline(x=50, color='orange', linestyle='--', label='50% threshold')
plt.axvline(x=90, color='red',    linestyle='--', label='90% threshold')
plt.xlabel('Missing %')
plt.title('Top 30 Columns with Missing Values', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("FEATURE TYPE BREAKDOWN")
print("=" * 60)

numerical_cols   = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n🔢 Numerical Features  : {len(numerical_cols)}")
print(f"🔤 Categorical Features: {len(categorical_cols)}")
print(f"\n📋 Categorical columns : {categorical_cols}")

In [ ]:
print("=" * 60)
print("TRANSACTION AMOUNT ANALYSIS")
print("=" * 60)

print("\nTransaction Amount Stats:")
print(df.groupby('isFraud')['TransactionAmt'].describe())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution (clipped)
axes[0].hist(df[df['isFraud']==0]['TransactionAmt'].clip(upper=1000),
             bins=50, alpha=0.6, color='#2ecc71', label='Legitimate')
axes[0].hist(df[df['isFraud']==1]['TransactionAmt'].clip(upper=1000),
             bins=50, alpha=0.6, color='#e74c3c', label='Fraud')
axes[0].set_title('Transaction Amount Distribution', fontweight='bold')
axes[0].set_xlabel('Amount (clipped at $1000)')
axes[0].legend()

# Log scale
axes[1].hist(np.log1p(df[df['isFraud']==0]['TransactionAmt']),
             bins=50, alpha=0.6, color='#2ecc71', label='Legitimate')
axes[1].hist(np.log1p(df[df['isFraud']==1]['TransactionAmt']),
             bins=50, alpha=0.6, color='#e74c3c', label='Fraud')
axes[1].set_title('Transaction Amount (Log Scale)', fontweight='bold')
axes[1].set_xlabel('log(Amount)')
axes[1].legend()

# Boxplot
df.boxplot(column='TransactionAmt', by='isFraud', ax=axes[2],
           patch_artist=True)
axes[2].set_title('Amount by Fraud Status', fontweight='bold')
axes[2].set_xlabel('isFraud (0=Legit, 1=Fraud)')
axes[2].set_ylim(0, 2000)

plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("TIME-BASED FRAUD PATTERN ANALYSIS")
print("=" * 60)

# TransactionDT = seconds from a reference point
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day']  = (df['TransactionDT'] // (3600 * 24)) % 7


fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Fraud rate by hour
hourly_fraud = df.groupby('hour')['isFraud'].mean() * 100
axes[0].plot(hourly_fraud.index, hourly_fraud.values,
             marker='o', color='#e74c3c', linewidth=2)
axes[0].fill_between(hourly_fraud.index, hourly_fraud.values,
                     alpha=0.2, color='#e74c3c')
axes[0].set_title('Fraud Rate by Hour of Day', fontweight='bold')
axes[0].set_xlabel('Hour (0-23)')
axes[0].set_ylabel('Fraud Rate (%)')
axes[0].axhline(y=df['isFraud'].mean()*100,
                color='blue', linestyle='--', label='Overall avg')
axes[0].legend()

# Fraud rate by day
daily_fraud = df.groupby('day')['isFraud'].mean() * 100
days_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
axes[1].bar(range(7), daily_fraud.values, color='#3498db', edgecolor='black')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(days_labels)
axes[1].set_title('Fraud Rate by Day of Week', fontweight='bold')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].axhline(y=df['isFraud'].mean()*100,
                color='red', linestyle='--', label='Overall avg')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("CATEGORICAL FEATURE ANALYSIS")
print("=" * 60)

cat_cols_to_plot = ['ProductCD', 'card4', 'card6', 'P_emaildomain']
cat_cols_to_plot = [c for c in cat_cols_to_plot if c in df.columns]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols_to_plot):
    # Only plot top 10 categories for readability
    top_cats = df[col].value_counts().head(10).index
    subset = df[df[col].isin(top_cats)]
    fraud_rate = subset.groupby(col)['isFraud'].mean().sort_values(ascending=False) * 100
    fraud_rate.plot(kind='bar', ax=axes[i],
                    color='#e74c3c', edgecolor='black', alpha=0.8)
    axes[i].set_title(f'Fraud Rate by {col}', fontweight='bold')
    axes[i].set_ylabel('Fraud Rate (%)')
    axes[i].set_xlabel(col)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].axhline(y=df['isFraud'].mean()*100,
                    color='blue', linestyle='--', label='Overall avg')
    axes[i].legend()

plt.suptitle('Fraud Rate by Categorical Features', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("CORRELATION WITH TARGET (isFraud)")
print("=" * 60)

# Top correlated numerical features with target
corr_with_target = (df[numerical_cols]
                    .corr()['isFraud']
                    .abs()
                    .sort_values(ascending=False)
                    .drop('isFraud')
                    .head(20))

print("\nTop 20 features most correlated with isFraud:")
print(corr_with_target.to_string())

plt.figure(figsize=(12, 6))
corr_with_target.plot(kind='bar', color='#3498db', edgecolor='black')
plt.title('Top 20 Features Correlated with Fraud', fontweight='bold')
plt.ylabel('Absolute Correlation')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()